## Multimodal Alignment Training Loop

In this Milestone 11, I train CLIP‑style multimodal model so that molecule embeddings and phenotype image embeddings begin to align in a shared representation space.

### Why this step matters
Before using real biological data, I must verify that:
- The training loop runs correctly
- The multimodal model produces embeddings
- The contrastive loss computes without errors
- Backpropagation updates model weights
- The notebook structure is correct

To keep things simple, I will use **dummy data** (random tensors) instead of real molecule–image pairs. This allows to test the training pipeline end‑to‑end without worrying about data loading.

### What happens in this step
1. Generate fake molecule and image embeddings using `torch.randn`.
2. Build the encoders and multimodal model defined in earlier steps.
3. Create an optimizer (Adam).
4. Run a small training loop:
   - Forward pass through the multimodal model  
   - Compute CLIP‑style contrastive loss  
   - Backpropagate and update weights  
   - Track average loss per epoch  

Because the data is random, the model cannot learn meaningful relationships.  
Therefore, the loss will hover around a stable value (usually between 2.0 and 3.0).  
This is **expected** and confirms the training loop is functioning correctly.

### Outcome
By the end of this milestone 11:
- The multimodal training pipeline is fully operational.
- All components (encoders, projection heads, contrastive loss, optimizer) work together.
- Ready to replace the dummy data with real molecule–phenotype pairs in Milestone 12 and 13.

This step ensures your foundation is solid before moving on to real multimodal learning.


In [25]:
# ------------------------------------------------------------
# Core PyTorch imports
# ------------------------------------------------------------

import torch                     # Main PyTorch library (tensors, GPU support)
import torch.nn as nn            # Neural network layers (Linear, Conv, etc.)
import torch.optim as optim      # Optimizers (Adam, SGD, etc.)
import torch.nn.functional as F  # Activation functions (ReLU, softmax, etc.)

# ------------------------------------------------------------
# Vision models (ResNet18) for the ImageEncoder
# ------------------------------------------------------------

from torchvision import models    # Gives access to pretrained CNN architectures

# ------------------------------------------------------------
# RDKit for molecule handling (SMILES → Mol objects)
# ------------------------------------------------------------

from rdkit import Chem            # Chemical structure parsing and manipulation




In [26]:
class SimpleMolEncoder(nn.Module):
    def __init__(self, max_atomic_num=100, emb_dim=256):
        super().__init__()
        # A simple linear layer that maps a 100‑dim atom count vector
        # into a 256‑dim molecular embedding.
        self.fc = nn.Linear(max_atomic_num, emb_dim)

    def forward(self, mol):
        # Create a zero vector to count atoms by atomic number (Z).
        # Index 0–99 correspond to atomic numbers 0–99.
        atom_counts = torch.zeros(100)

        # Loop through each atom in the molecule
        for atom in mol.GetAtoms():
            Z = atom.GetAtomicNum()       # atomic number of the atom
            if Z < 100:                   # only count atoms within our vector size
                atom_counts[Z] += 1.0     # increment count for that atomic number

        # Add batch dimension: shape becomes (1, 100)
        atom_counts = atom_counts.unsqueeze(0)

        # Pass the atom count vector through the linear layer
        # and apply ReLU to get a 256‑dim embedding
        emb = F.relu(self.fc(atom_counts))

        # Return the molecular embedding
        return emb



In [27]:
class ImageEncoder(nn.Module):
    def __init__(self, emb_dim=256):
        super().__init__()

        # Load a pretrained ResNet18 model.
        # This gives us a strong image feature extractor without training from scratch.
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # Get the number of features produced by the final ResNet layer.
        # We need this to replace the classification head with our own projection layer.
        in_features = self.backbone.fc.in_features

        # Replace ResNet18's final fully-connected layer with a new Linear layer
        # that outputs an embedding of size emb_dim (default = 256).
        # This makes the image embedding compatible with the molecule embedding size.
        self.backbone.fc = nn.Linear(in_features, emb_dim)

    def forward(self, img):
        # If the input image has shape (C, H, W), add a batch dimension
        # so it becomes (1, C, H, W). ResNet requires a batch dimension.
        if img.dim() == 3:
            img = img.unsqueeze(0)

        # Pass the image through the ResNet backbone and apply ReLU activation.
        # This produces a 256-dimensional image embedding.
        emb = F.relu(self.backbone(img))

        # Return the final image embedding.
        return emb


In [28]:
class MultimodalModel(nn.Module):
    def __init__(self, emb_dim=256, proj_dim=128):
        super().__init__()

        # Linear projection layer for molecule embeddings.
        # Takes a 256‑dim input (emb_dim) and maps it into a 128‑dim shared space (proj_dim).
        self.mol_proj = nn.Linear(emb_dim, proj_dim)

        # Linear projection layer for image embeddings.
        # Ensures both modalities end up in the same shared embedding space.
        self.img_proj = nn.Linear(emb_dim, proj_dim)

    def forward(self, mol_emb, img_emb):
        # Project molecule embeddings and L2-normalize them.
        # Normalization ensures embeddings lie on a unit hypersphere,
        # which stabilizes contrastive learning.
        z_mol = F.normalize(self.mol_proj(mol_emb), dim=1)

        # Project image embeddings and normalize them in the same way.
        z_img = F.normalize(self.img_proj(img_emb), dim=1)

        # Return both projected embeddings.
        return z_mol, z_img

    def contrastive_loss(self, z_mol, z_img, temperature=0.1):
        # Compute pairwise similarity between all molecule and image embeddings.
        # This produces a matrix of shape (batch_size, batch_size).
        # Dividing by temperature sharpens or smooths the similarity distribution.
        sim_matrix = torch.matmul(z_mol, z_img.T) / temperature

        # Number of samples in the batch.
        batch_size = sim_matrix.size(0)

        # Labels represent the "correct" pairing:
        # molecule i should match image i.
        labels = torch.arange(batch_size).to(sim_matrix.device)

        # Cross-entropy loss for molecule → image direction.
        # Each molecule embedding should match its corresponding image embedding.
        loss_mol_to_img = nn.CrossEntropyLoss()(sim_matrix, labels)

        # Cross-entropy loss for image → molecule direction.
        # Each image embedding should match its corresponding molecule embedding.
        loss_img_to_mol = nn.CrossEntropyLoss()(sim_matrix.T, labels)

        # Final contrastive loss is the average of both directions.
        return (loss_mol_to_img + loss_img_to_mol) / 2


In [29]:
# Instantiate the molecule encoder.
# This creates an instance of the SimpleMolEncoder class defined earlier.
# It will convert RDKit molecule objects into 256‑dimensional embeddings.
mol_encoder = SimpleMolEncoder()

# Instantiate the image encoder.
# This creates an instance of the ImageEncoder class defined earlier.
# It uses a pretrained ResNet18 backbone to convert images into 256‑dimensional embeddings.
img_encoder = ImageEncoder()


In [30]:
# Create an instance of the multimodal model.
# This model contains:
# - A projection head for molecule embeddings
# - A projection head for image embeddings
# - A contrastive loss function
# It aligns both modalities into the same shared embedding space.
model = MultimodalModel()

# Set up the Adam optimizer.
# It will update all trainable parameters inside `model`
# using a learning rate of 1e-3 (a common default for beginners).
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [31]:
# Print a simple message to indicate that the training loop is about to begin.
# This helps you visually separate setup steps from the actual training output.
print("Starting multimodal alignment training\n")


Starting multimodal alignment training



In [ ]:
# Loop over 5 training epochs
for epoch in range(5):

    # Track total loss for this epoch so we can compute the average later
    total_loss = 0.0

    # Iterate over all batches in the epoch
    for _ in range(num_batches):

        # ------------------------------------------------------------
        # 1. Generate a batch of fake molecule + image embeddings
        # ------------------------------------------------------------
        # These are random tensors used only to test the training pipeline.
        mol_emb, img_emb = generate_fake_batch(batch_size)

        # ------------------------------------------------------------
        # 2. Forward pass through the multimodal model
        # ------------------------------------------------------------
        # The model projects both embeddings into a shared space.
        z_mol, z_img = model(mol_emb, img_emb)

        # ------------------------------------------------------------
        # 3. Compute CLIP-style contrastive loss
        # ------------------------------------------------------------
        # This encourages matching molecule–image pairs to be similar.
        loss = model.contrastive_loss(z_mol, z_img)

        # ------------------------------------------------------------
        # 4. Backpropagation
        # ------------------------------------------------------------
        optimizer.zero_grad()   # Reset gradients from previous step
        loss.backward()         # Compute gradients for current loss
        optimizer.step()        # Update model weights

        # Add this batch's loss to the running total
        total_loss += loss.item()

    # ------------------------------------------------------------
    # 5. Compute average loss for the epoch
    # ------------------------------------------------------------
    avg_loss = total_loss / num_batches

    # Print progress so you can monitor training behaviour
    print(f"Epoch {epoch+1} — Average Loss: {avg_loss:.4f}")

    # Message to visually separate epochs
    print("\nTraining complete.")


Epoch 1 — Average Loss: 2.3968

Training complete.
Epoch 2 — Average Loss: 2.4233

Training complete.
Epoch 3 — Average Loss: 2.4325

Training complete.
Epoch 4 — Average Loss: 2.3755

Training complete.
Epoch 5 — Average Loss: 2.5027

Training complete.
